# Lab 06 External V2 — 02 Fact Encounters

**Dataset:** Synthea Healthcare  
**Gold fact grain:** **one row per healthcare encounter**

## Purpose

Build `fact_encounters` from the Synthea `encounters.csv` source and resolve
foreign keys to the conformed Gold dimensions:

- `dim_date`
- `dim_patient`
- `dim_provider`
- `dim_organization`
- `dim_payer`

Measures include:

- encounter duration,
- base encounter cost,
- total claim cost,
- payer coverage,
- patient responsibility.

### Design choices

- The full historical encounter source is used for the Gold fact.
- Monthly landing partitions remain an operational/alert simulation concern.
- Business identifiers are retained for traceability.
- Surrogate dimension keys are resolved through joins.
- Environment-specific values come from parameters.
- No `current_user()` paths.

## 1. Runtime context

In [ ]:
import sys
from pathlib import Path

lab_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(lab_root) not in sys.path:
    sys.path.insert(0, str(lab_root))

from src.runtime_config import load_runtime_context


ctx = load_runtime_context(
    dbutils,
    include_validation=True,
)

config = ctx.config
run_validation = ctx.run_validation

print(f"Source volume   : {config.source_volume_path}")
print(f"Target schema   : {config.target_schema_fqn}")
print(f"External root   : {config.external_gold_root}")
print(f"Run validation : {run_validation}")

## 2. Shared configuration

In [ ]:
from src.external_tables import write_external_delta

encounters_source = f"{config.source_csv_path}/encounters.csv"

print(f"Source : {encounters_source}")
print(f"Target : {config.fact_encounters}")
print(f"Path   : {config.table_path('fact_encounters')}")

## 3. Imports and source load

In [ ]:
from pyspark.sql import functions as F

encounters_src = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(encounters_source)
)

print(f"Source rows: {encounters_src.count():,}")
print(f"Source columns: {len(encounters_src.columns)}")

display(encounters_src.limit(10))

## 4. Validate encounter source schema

In [ ]:
REQUIRED_COLUMNS = {
    "Id",
    "START",
    "STOP",
    "PATIENT",
    "ORGANIZATION",
    "PROVIDER",
    "PAYER",
    "ENCOUNTERCLASS",
    "CODE",
    "DESCRIPTION",
    "BASE_ENCOUNTER_COST",
    "TOTAL_CLAIM_COST",
    "PAYER_COVERAGE",
    "REASONCODE",
    "REASONDESCRIPTION",
}

missing_columns = sorted(
    REQUIRED_COLUMNS - set(encounters_src.columns)
)

if missing_columns:
    raise ValueError(
        "encounters.csv is missing required columns: "
        + ", ".join(missing_columns)
    )

print("Encounter source schema validation passed.")

## 5. Prepare encounter business columns and measures

In [ ]:
prepared_encounters = (
    encounters_src
    .select(
        F.col("Id").alias("encounter_id"),
        F.to_timestamp("START").alias("encounter_start_ts"),
        F.to_timestamp("STOP").alias("encounter_stop_ts"),
        F.col("PATIENT").alias("patient_id"),
        F.col("ORGANIZATION").alias("organization_id"),
        F.col("PROVIDER").alias("provider_id"),
        F.col("PAYER").alias("payer_id"),
        F.col("ENCOUNTERCLASS").alias("encounter_class"),
        F.col("CODE").alias("encounter_code"),
        F.col("DESCRIPTION").alias("encounter_description"),
        F.col("BASE_ENCOUNTER_COST")
            .cast("decimal(18,2)")
            .alias("base_encounter_cost"),
        F.col("TOTAL_CLAIM_COST")
            .cast("decimal(18,2)")
            .alias("total_claim_cost"),
        F.col("PAYER_COVERAGE")
            .cast("decimal(18,2)")
            .alias("payer_coverage"),
        F.col("REASONCODE").alias("reason_code"),
        F.col("REASONDESCRIPTION").alias("reason_description"),
    )
    .withColumn(
        "encounter_date",
        F.to_date("encounter_start_ts"),
    )
    .withColumn(
        "duration_minutes",
        (
            F.unix_timestamp("encounter_stop_ts")
            - F.unix_timestamp("encounter_start_ts")
        ) / F.lit(60.0),
    )
    .withColumn(
        "patient_responsibility",
        (
            F.col("total_claim_cost")
            - F.col("payer_coverage")
        ).cast("decimal(18,2)"),
    )
)

display(prepared_encounters.limit(10))

## 6. Validate source grain and core measures before dimension joins

In [ ]:
source_quality_row = (
    prepared_encounters
    .agg(
        F.count("*").alias("row_count"),
        F.countDistinct("encounter_id").alias("distinct_encounter_ids"),
        F.sum(
            F.when(F.col("encounter_id").isNull(), 1).otherwise(0)
        ).alias("null_encounter_ids"),
        F.sum(
            F.when(F.col("encounter_start_ts").isNull(), 1).otherwise(0)
        ).alias("invalid_start_rows"),
        F.sum(
            F.when(F.col("encounter_stop_ts").isNull(), 1).otherwise(0)
        ).alias("invalid_stop_rows"),
        F.sum(
            F.when(F.col("duration_minutes") < 0, 1).otherwise(0)
        ).alias("negative_duration_rows"),
        F.sum(
            F.when(F.col("total_claim_cost") < 0, 1).otherwise(0)
        ).alias("negative_claim_rows"),
    )
    .first()
)

source_quality_df = spark.createDataFrame(
    [(
        int(source_quality_row["row_count"]),
        int(source_quality_row["distinct_encounter_ids"]),
        int(source_quality_row["null_encounter_ids"] or 0),
        int(source_quality_row["invalid_start_rows"] or 0),
        int(source_quality_row["invalid_stop_rows"] or 0),
        int(source_quality_row["negative_duration_rows"] or 0),
        int(source_quality_row["negative_claim_rows"] or 0),
    )],
    [
        "row_count",
        "distinct_encounter_ids",
        "null_encounter_ids",
        "invalid_start_rows",
        "invalid_stop_rows",
        "negative_duration_rows",
        "negative_claim_rows",
    ],
)

display(source_quality_df)

source_failures = []

if source_quality_row["row_count"] != source_quality_row["distinct_encounter_ids"]:
    source_failures.append("duplicate encounter_id")

if (source_quality_row["null_encounter_ids"] or 0) > 0:
    source_failures.append("null encounter_id")

if (source_quality_row["invalid_start_rows"] or 0) > 0:
    source_failures.append("invalid START timestamp")

if (source_quality_row["invalid_stop_rows"] or 0) > 0:
    source_failures.append("invalid STOP timestamp")

if (source_quality_row["negative_duration_rows"] or 0) > 0:
    source_failures.append("negative duration")

if (source_quality_row["negative_claim_rows"] or 0) > 0:
    source_failures.append("negative claim cost")

if source_failures:
    raise ValueError(
        "Encounter source quality validation failed: "
        + ", ".join(source_failures)
    )

print("Encounter source grain and measure validation passed.")

## 7. Load Gold dimensions

In [ ]:
dim_date = (
    spark.table(config.dim_date)
    .select(
        "date_key",
        "full_date",
    )
)

dim_patient = (
    spark.table(config.dim_patient)
    .select(
        "patient_key",
        "patient_id",
    )
)

dim_provider = (
    spark.table(config.dim_provider)
    .select(
        "provider_key",
        "provider_id",
    )
)

dim_organization = (
    spark.table(config.dim_organization)
    .select(
        "organization_key",
        "organization_id",
    )
)

dim_payer = (
    spark.table(config.dim_payer)
    .select(
        "payer_key",
        "payer_id",
    )
)

print("Gold dimensions loaded.")

## 8. Resolve dimension foreign keys

In [ ]:
fact_encounters_df = (
    prepared_encounters.alias("e")
    .join(
        dim_date.alias("d"),
        F.col("e.encounter_date") == F.col("d.full_date"),
        "left",
    )
    .join(
        dim_patient.alias("pat"),
        F.col("e.patient_id") == F.col("pat.patient_id"),
        "left",
    )
    .join(
        dim_provider.alias("pr"),
        F.col("e.provider_id") == F.col("pr.provider_id"),
        "left",
    )
    .join(
        dim_organization.alias("org"),
        F.col("e.organization_id") == F.col("org.organization_id"),
        "left",
    )
    .join(
        dim_payer.alias("pay"),
        F.col("e.payer_id") == F.col("pay.payer_id"),
        "left",
    )
    .select(
        F.xxhash64("e.encounter_id").alias("encounter_key"),
        F.col("e.encounter_id"),

        F.col("d.date_key"),
        F.col("pat.patient_key"),
        F.col("pr.provider_key"),
        F.col("org.organization_key"),
        F.col("pay.payer_key"),

        F.col("e.patient_id"),
        F.col("e.provider_id"),
        F.col("e.organization_id"),
        F.col("e.payer_id"),

        F.col("e.encounter_start_ts"),
        F.col("e.encounter_stop_ts"),
        F.col("e.encounter_date"),

        F.col("e.encounter_class"),
        F.col("e.encounter_code"),
        F.col("e.encounter_description"),

        F.col("e.base_encounter_cost"),
        F.col("e.total_claim_cost"),
        F.col("e.payer_coverage"),
        F.col("e.patient_responsibility"),
        F.round(F.col("e.duration_minutes"), 2).alias("duration_minutes"),

        F.col("e.reason_code"),
        F.col("e.reason_description"),
    )
)

display(fact_encounters_df.limit(10))

## 9. Foreign-key validation

Required FK relationships must resolve for:

- date,
- patient,
- organization,
- payer.

`provider_key` is validated separately because some synthetic encounter
records can legitimately have no populated provider identifier.

In [ ]:
fk_validation_row = (
    fact_encounters_df
    .agg(
        F.count("*").alias("row_count"),
        F.sum(
            F.when(F.col("date_key").isNull(), 1).otherwise(0)
        ).alias("missing_date_fk"),
        F.sum(
            F.when(F.col("patient_key").isNull(), 1).otherwise(0)
        ).alias("missing_patient_fk"),
        F.sum(
            F.when(F.col("organization_key").isNull(), 1).otherwise(0)
        ).alias("missing_organization_fk"),
        F.sum(
            F.when(F.col("payer_key").isNull(), 1).otherwise(0)
        ).alias("missing_payer_fk"),
        F.sum(
            F.when(
                F.col("provider_id").isNotNull()
                & F.col("provider_key").isNull(),
                1,
            ).otherwise(0)
        ).alias("unresolved_provider_fk"),
        F.sum(
            F.when(F.col("provider_id").isNull(), 1).otherwise(0)
        ).alias("source_null_provider_id"),
    )
    .first()
)

fk_validation_df = spark.createDataFrame(
    [(
        int(fk_validation_row["row_count"]),
        int(fk_validation_row["missing_date_fk"] or 0),
        int(fk_validation_row["missing_patient_fk"] or 0),
        int(fk_validation_row["missing_organization_fk"] or 0),
        int(fk_validation_row["missing_payer_fk"] or 0),
        int(fk_validation_row["unresolved_provider_fk"] or 0),
        int(fk_validation_row["source_null_provider_id"] or 0),
    )],
    [
        "row_count",
        "missing_date_fk",
        "missing_patient_fk",
        "missing_organization_fk",
        "missing_payer_fk",
        "unresolved_provider_fk",
        "source_null_provider_id",
    ],
)

display(fk_validation_df)

required_fk_failures = []

for field in [
    "missing_date_fk",
    "missing_patient_fk",
    "missing_organization_fk",
    "missing_payer_fk",
    "unresolved_provider_fk",
]:
    if (fk_validation_row[field] or 0) > 0:
        required_fk_failures.append(field)

if run_validation and required_fk_failures:
    raise ValueError(
        "Fact foreign-key validation failed: "
        + ", ".join(required_fk_failures)
    )

print("Fact foreign-key validation passed.")

## 10. Persist `fact_encounters`

In [ ]:
fact_path = write_external_delta(
    spark,
    fact_encounters_df,
    config,
    config.fact_encounters,
)

print(f"Created external table: {config.fact_encounters}")
print(f"Physical Delta path   : {fact_path}")

## 11. Fact grain reconciliation

In [ ]:
source_count = prepared_encounters.count()

target_profile = (
    spark.table(config.fact_encounters)
    .agg(
        F.count("*").alias("target_count"),
        F.countDistinct("encounter_id").alias("distinct_encounter_ids"),
        F.countDistinct("encounter_key").alias("distinct_encounter_keys"),
    )
    .first()
)

target_count = int(target_profile["target_count"])
distinct_encounter_ids = int(target_profile["distinct_encounter_ids"])
distinct_encounter_keys = int(target_profile["distinct_encounter_keys"])

grain_status = (
    "PASS"
    if (
        target_count == source_count
        and target_count == distinct_encounter_ids
        and target_count == distinct_encounter_keys
    )
    else "FAIL"
)

grain_validation_df = spark.createDataFrame(
    [(
        source_count,
        target_count,
        distinct_encounter_ids,
        distinct_encounter_keys,
        grain_status,
    )],
    [
        "source_rows",
        "fact_rows",
        "distinct_encounter_ids",
        "distinct_encounter_keys",
        "status",
    ],
)

display(grain_validation_df)

if run_validation and grain_status == "FAIL":
    raise ValueError(
        "fact_encounters grain reconciliation failed."
    )

print("fact_encounters grain reconciliation passed.")

## 12. Measure reconciliation

In [ ]:
source_measure_profile = (
    prepared_encounters
    .agg(
        F.sum("base_encounter_cost").alias("base_encounter_cost"),
        F.sum("total_claim_cost").alias("total_claim_cost"),
        F.sum("payer_coverage").alias("payer_coverage"),
        F.sum("patient_responsibility").alias("patient_responsibility"),
    )
    .first()
)

target_measure_profile = (
    spark.table(config.fact_encounters)
    .agg(
        F.sum("base_encounter_cost").alias("base_encounter_cost"),
        F.sum("total_claim_cost").alias("total_claim_cost"),
        F.sum("payer_coverage").alias("payer_coverage"),
        F.sum("patient_responsibility").alias("patient_responsibility"),
    )
    .first()
)

measure_rows = []

for measure in [
    "base_encounter_cost",
    "total_claim_cost",
    "payer_coverage",
    "patient_responsibility",
]:
    source_value = source_measure_profile[measure]
    target_value = target_measure_profile[measure]

    status = "PASS" if source_value == target_value else "FAIL"

    measure_rows.append(
        (
            measure,
            str(source_value),
            str(target_value),
            status,
        )
    )

measure_validation_df = spark.createDataFrame(
    measure_rows,
    [
        "measure",
        "source_total",
        "fact_total",
        "status",
    ],
)

display(measure_validation_df)

failed_measures = [
    row["measure"]
    for row in measure_validation_df.collect()
    if row["status"] == "FAIL"
]

if run_validation and failed_measures:
    raise ValueError(
        "Fact measure reconciliation failed: "
        + ", ".join(failed_measures)
    )

print("Fact measure reconciliation passed.")

## 13. Business profile

In [ ]:
fact_business_profile_df = (
    spark.table(config.fact_encounters)
    .groupBy("encounter_class")
    .agg(
        F.count("*").alias("encounter_count"),
        F.countDistinct("patient_key").alias("unique_patients"),
        F.round(
            F.avg("duration_minutes"),
            2,
        ).alias("avg_duration_minutes"),
        F.round(
            F.sum("total_claim_cost"),
            2,
        ).alias("total_claim_cost"),
        F.round(
            F.avg("total_claim_cost"),
            2,
        ).alias("avg_claim_cost"),
    )
    .orderBy(
        F.desc("encounter_count")
    )
)

display(fact_business_profile_df)

## 14. Final validation summary

In [ ]:
final_checks = [
    ("source_grain", len(source_failures) == 0),
    ("foreign_keys", len(required_fk_failures) == 0),
    ("fact_grain", grain_status == "PASS"),
    ("measure_reconciliation", len(failed_measures) == 0),
]

final_validation_df = spark.createDataFrame(
    [
        (
            check_name,
            "PASS" if passed else "FAIL",
        )
        for check_name, passed in final_checks
    ],
    ["check_name", "status"],
)

display(final_validation_df)

failed_final_checks = [
    name
    for name, passed in final_checks
    if not passed
]

if run_validation and failed_final_checks:
    raise RuntimeError(
        "Final fact validation failed: "
        + ", ".join(failed_final_checks)
    )

## 15. Completion

`fact_encounters` grain:

> **One row per healthcare encounter.**

Key relationships:

```text
                     dim_date
                        |
                        |
dim_patient ─── fact_encounters ─── dim_provider
                        |
             ┌──────────┴──────────┐
             |                     |
      dim_organization         dim_payer
```

**Next:** build `fact_conditions`.

In [ ]:
print("LAB 06 — FACT ENCOUNTERS COMPLETE")
print("")
print(f"Created: {config.fact_encounters}")
print("")
print("Next: lab06_03_fact_conditions")